# Clustering de outlets - TFM Altadis## Objetivo 4: Clusterización de datosEste notebook segmenta los 2.957 outlets con tamaño conocido (Tam_m2 != 'N.D.') en grupos según su perfil de negocio, combinando variables descriptivas (tamaño, renta provincial) con variables de comportamiento de venta (volumen, rotura de stock, diversidad de catálogo).**Fichero de entrada esperado:** `TablaClustering.csv` (exportado desde SQL Server, separador `;`, decimales con coma, sin cabecera).

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scoreimport matplotlib.pyplot as pltpd.set_option('display.width', 120)

## 1. Carga y limpieza de datosEl fichero exportado desde SQL Server mezcla dos formatos de decimal: la columna `Tam_m2_Numerico` usa punto (`7.5`, `15.0`) mientras que el resto usa coma (formato español de Excel, `0,0035...`). Se corrige explícitamente.

In [ ]:
cols = ['Affiliated_Code','Provincia','Location','Tam_m2_Numerico',        'Renta_Media_Provincial','Ventas_Totales','Tasa_Rotura','Num_Productos_Distintos']df = pd.read_csv('TablaClustering.csv', sep=';', decimal=',',                  header=None, names=cols, encoding='utf-8-sig')# Tam_m2_Numerico usa punto decimal, no coma: se corrige por separadodf['Tam_m2_Numerico'] = df['Tam_m2_Numerico'].astype(float)print(df.shape)print(df.dtypes)df.head()

In [ ]:
# Verificación de calidad: no debe haber nulosprint(df.isnull().sum())print()print(df.describe().round(2))

## 2. Selección de variablesVariables incluidas en el clustering: `Tam_m2_Numerico`, `Renta_Media_Provincial`, `Ventas_Totales`, `Tasa_Rotura`, `Num_Productos_Distintos`.Se excluyen `Provincia` (alta cardinalidad, parcialmente redundante con la renta) y `Location` (se reserva como variable de caracterización posterior, no de entrada al algoritmo, para poder usarla luego como lectura cualitativa de cada cluster).Las variables se estandarizan (media 0, desviación 1) antes de aplicar k-means, ya que están en escalas muy distintas (la renta está en miles de euros, la tasa de rotura en decimales entre 0 y 1).

In [ ]:
features = ['Tam_m2_Numerico','Renta_Media_Provincial','Ventas_Totales',            'Tasa_Rotura','Num_Productos_Distintos']X = df[features].valuesscaler = StandardScaler()X_scaled = scaler.fit_transform(X)

## 3. Elección del número de clustersSe evalúan los valores de k entre 2 y 8 mediante el método del codo (inercia) y el coeficiente de silueta (silhouette score), para justificar la elección de k de forma objetiva en lugar de arbitraria.

In [ ]:
inertias = []silhouettes = []K_range = range(2, 9)for k in K_range:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(X_scaled)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X_scaled, labels))resultados_k = pd.DataFrame({'k': list(K_range), 'Inercia': inertias, 'Silhouette': silhouettes})print(resultados_k)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))axes[0].plot(K_range, inertias, marker='o', color='#2D5F9E')axes[0].set_xlabel('Número de clusters (k)')axes[0].set_ylabel('Inercia')axes[0].set_title('Método del codo')axes[0].grid(alpha=0.3)axes[1].plot(K_range, silhouettes, marker='o', color='#C0521F')axes[1].set_xlabel('Número de clusters (k)')axes[1].set_ylabel('Silhouette score')axes[1].set_title('Coeficiente de silueta')axes[1].grid(alpha=0.3)plt.tight_layout()plt.savefig('Clustering_Eleccion_K.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

**Decisión:** se elige **k=5**. El silhouette score máximo se alcanza en k=7 (0,254), pero la diferencia frente a k=5 (0,233) es reducida, y un número menor de clusters resulta más manejable para la interpretación de negocio y la presentación de resultados en el TFM, sin pérdida sustancial de calidad del agrupamiento.

## 4. Primera ejecución con k=5: detección de un cluster degeneradoUna primera ejecución de k-means con k=5 sobre el conjunto completo (2.957 outlets) produjo un cluster de únicamente 5 observaciones, lo que indica un grupo de outliers extremos más que un segmento de negocio real.

In [ ]:
km_inicial = KMeans(n_clusters=5, random_state=42, n_init=10)df['Cluster_Inicial'] = km_inicial.fit_predict(X_scaled)print(df['Cluster_Inicial'].value_counts().sort_index())

In [ ]:
# Inspección del cluster minoritariocluster_pequeno = df['Cluster_Inicial'].value_counts().idxmin()outliers = df[df['Cluster_Inicial'] == cluster_pequeno]print(f"Outlets en el cluster degenerado: {len(outliers)}")print(outliers[features + ['Affiliated_Code','Provincia']])print()print("Media general de Tasa_Rotura:", df['Tasa_Rotura'].mean())print("Tasa_Rotura de estos outlets:", outliers['Tasa_Rotura'].values)

**Hallazgo:** estos 5 outlets tienen una tasa de rotura de stock entre 50 y 90 veces superior a la media de la red (0,05–0,086 frente a 0,0009 de media general). No constituyen un segmento de negocio por su reducido tamaño muestral, sino un hallazgo cualitativo independiente: outlets con un problema de suministro extremo y aislado que merece atención prioritaria, documentado aparte del análisis de clustering principal.Se excluyen del clustering y se repite el proceso sobre los 2.952 outlets restantes.

In [ ]:
df_sin_outliers = df[df['Cluster_Inicial'] != cluster_pequeno].copy()print("Outlets tras excluir los outliers extremos de rotura:", len(df_sin_outliers))X2 = df_sin_outliers[features].valuesscaler2 = StandardScaler()X2_scaled = scaler2.fit_transform(X2)# Se reconfirma que k=5 sigue siendo razonable sin los outliersfor k in [4,5,6]:    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km_test.fit_predict(X2_scaled)    sil = silhouette_score(X2_scaled, labels)    print(f"k={k}: silhouette={sil:.4f}, tamaños={np.bincount(labels)}")

## 5. Clustering final (k=5, sin outliers de rotura)

In [ ]:
km_final = KMeans(n_clusters=5, random_state=42, n_init=10)df_sin_outliers['Cluster'] = km_final.fit_predict(X2_scaled)print(df_sin_outliers['Cluster'].value_counts().sort_index())

In [ ]:
perfil = df_sin_outliers.groupby('Cluster')[features].mean().round(2)perfil['N_Outlets'] = df_sin_outliers['Cluster'].value_counts().sort_index()print(perfil)

In [ ]:
# Caracterización cualitativa: tipo de Location dominante por clusterfor c in sorted(df_sin_outliers['Cluster'].unique()):    top_loc = df_sin_outliers[df_sin_outliers['Cluster']==c]['Location'].value_counts().head(2)    print(f"Cluster {c}: {dict(top_loc)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))colors = ['#2D5F9E','#C0521F','#1C7A5C','#8B3A8B','#B8941F']for c in sorted(df_sin_outliers['Cluster'].unique()):    sub = df_sin_outliers[df_sin_outliers['Cluster']==c]    axes[0].scatter(sub['Tam_m2_Numerico'], sub['Ventas_Totales'],                     s=14, alpha=0.55, color=colors[c], label=f'Cluster {c} (n={len(sub)})')axes[0].set_xlabel('Tamaño del local (m², codificado)')axes[0].set_ylabel('Ventas totales (unidades)')axes[0].set_title('Tamaño vs. Volumen de venta')axes[0].legend(fontsize=8)axes[0].grid(alpha=0.25)for c in sorted(df_sin_outliers['Cluster'].unique()):    sub = df_sin_outliers[df_sin_outliers['Cluster']==c]    axes[1].scatter(sub['Renta_Media_Provincial'], sub['Ventas_Totales'],                     s=14, alpha=0.55, color=colors[c])axes[1].set_xlabel('Renta media provincial (€)')axes[1].set_ylabel('Ventas totales (unidades)')axes[1].set_title('Renta vs. Volumen de venta')axes[1].grid(alpha=0.25)plt.tight_layout()plt.savefig('Clustering_Resultado_5_Grupos.png', dpi=200, bbox_inches='tight', facecolor='white')plt.show()

## 6. Interpretación de negocio- **Cluster 0** (715 outlets): tamaño pequeño, renta provincial alta, venta media.- **Cluster 1** (959 outlets, el más numeroso): tamaño pequeño, renta baja, venta baja y catálogo reducido — el segmento de menor rendimiento comercial.- **Cluster 2** (727 outlets): venta notablemente superior al resto (casi el triple de la media) y mayor diversidad de catálogo — el segmento de mejor desempeño.- **Cluster 3** (193 outlets): perfil similar al promedio, pero con una tasa de rotura de stock más elevada que el resto de grupos (sin llegar a ser un caso extremo).- **Cluster 4** (358 outlets): único grupo diferenciado claramente por el tamaño físico del local (28 m² de media frente a 7-9 m² del resto), sin que ese mayor tamaño se traduzca en un volumen de venta proporcionalmente superior.**Conclusión principal:** el tamaño físico del local no es el factor que mejor explica las diferencias de rendimiento comercial entre outlets; el comportamiento de venta (volumen y diversidad de catálogo) separa los grupos con más claridad que las variables puramente descriptivas (tamaño, renta).

In [ ]:
# Exportar resultado para uso posterior (Power BI / informe)df_sin_outliers.to_csv('Outlets_con_Cluster.csv', index=False)outliers.to_csv('Outlets_Outliers_Rotura.csv', index=False)print("Ficheros exportados correctamente.")